# LifeMate: fine-tune Qwen3.5-4B (QLoRA) trên Colab, có thể đổi tài khoản giữa chừng

**Cách hoạt động:** mọi thứ cần để tiếp tục (dữ liệu, prompt, checkpoint) nằm trong **một repo Hugging Face PRIVATE** của bạn.
Cứ mỗi `SAVE_STEPS` step notebook đẩy checkpoint (adapter + optimizer + scheduler + RNG) lên `last-checkpoint/` của repo đó.
Colab bị ngắt thì mở notebook này ở **bất kỳ tài khoản Google nào**, nhập **cùng HF token**, chọn *Run all*: nó tự tải checkpoint về và train tiếp đúng step cũ.

**Lần đầu (1 lần duy nhất)**
1. Trên máy: `bash finetune/make_colab_bundle.sh` ra `finetune/out/lifemate_bundle.zip` (chỉ có `train.jsonl`, catalog, code; không có audio).
2. Tạo HF token loại *Write* tại huggingface.co/settings/tokens. Trong Colab: biểu tượng chìa khoá (Secrets) → thêm `HF_TOKEN`, bật *Notebook access*.
3. Runtime → Change runtime type → GPU (T4 trở lên). Chạy lần lượt các cell; cell 3 sẽ hỏi upload `lifemate_bundle.zip`.

**Tài khoản khác khi bị ngắt:** mở lại notebook này (upload `.ipynb` lên Drive của tài khoản đó), thêm cùng `HF_TOKEN`, Run all. Không cần upload lại gì.

**Sửa prompt:** xem cell 4. Đổi prompt = đổi dữ liệu train ⇒ phải train lại từ đầu (`FRESH_START = True`). Notebook từ chối resume nếu prompt/dữ liệu/tham số khác với checkpoint, tránh trộn hai phiên bản.

> Dữ liệu cuộc thi không được chia sẻ: giữ repo HF ở chế độ **Private** (notebook sẽ dừng nếu repo public).

In [ ]:
# 1. Cài thư viện. Nếu pip báo cần restart runtime thì restart rồi chạy tiếp từ cell 2.
!pip install -q -U "transformers>=5.0.0" peft accelerate bitsandbytes huggingface_hub pydantic

In [ ]:
# 2. Cấu hình + đăng nhập HF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                             # 1 GPU, phải đặt trước khi import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import getpass, hashlib, json, math, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from huggingface_hub import HfApi, login, hf_hub_download, snapshot_download

HF_REPO = None           # ví dụ "ten-hf/lifemate-qwen35-lora"; None = "<user HF>/lifemate-qwen35-lora"
FRESH_START = False      # True = xoá checkpoint trên HF và train lại từ đầu (bắt buộc sau khi đổi prompt)

MODEL_ID = "Qwen/Qwen3.5-4B"
MAX_LEN = 6144           # bỏ mẫu dài hơn; xem dòng "max len" ở cell 7. Nếu mất nhiều mẫu thì rút gọn prompt, đừng chỉ hạ số này
MAX_EXAMPLES = 300       # 300 = chạy thử ~20-40 phút. Đặt None để train đủ ~4377 example
EPOCHS, LR, ACCUM = 1, 1e-4, 16
LORA_R, LORA_ALPHA = 16, 32
SAVE_STEPS = 20          # cứ 20 step đẩy checkpoint lên HF (mất tối đa 20 step khi bị ngắt)
TIME_LIMIT_H = 11        # tự lưu + dừng trước khi hết giờ phiên
# TRAIN_CFG so khớp với checkpoint khi resume; đổi bất kỳ giá trị nào ở đây thì phải FRESH_START
TRAIN_CFG = dict(model=MODEL_ID, max_len=MAX_LEN, max_examples=MAX_EXAMPLES, epochs=EPOCHS, lr=LR, accum=ACCUM,
                 lora_r=LORA_R, lora_alpha=LORA_ALPHA, precision="fp16")

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (Write): ")
login(HF_TOKEN)
api = HfApi()
HF_REPO = HF_REPO or f"{api.whoami()['name']}/lifemate-qwen35-lora"
api.create_repo(HF_REPO, private=True, exist_ok=True)
assert api.repo_info(HF_REPO).private, f"{HF_REPO} đang PUBLIC: chuyển sang Private rồi chạy lại (dữ liệu cuộc thi không được phát tán)"
print("repo:", HF_REPO, "| files:", len(api.list_repo_files(HF_REPO)))

In [ ]:
# 3. Lấy bundle (code + train.jsonl) từ repo HF. Lần đầu sẽ hỏi upload lifemate_bundle.zip
WORK = Path("/content/lifemate")
repo_files = api.list_repo_files(HF_REPO)
if "bundle/lifemate_bundle.zip" not in repo_files:
    from google.colab import files as colab_files
    print("Chọn file lifemate_bundle.zip (tạo bằng finetune/make_colab_bundle.sh)")
    uploaded = colab_files.upload()
    api.upload_file(path_or_fileobj=next(iter(uploaded)), path_in_repo="bundle/lifemate_bundle.zip", repo_id=HF_REPO)
bundle = hf_hub_download(HF_REPO, "bundle/lifemate_bundle.zip")
shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir(parents=True)
zipfile.ZipFile(bundle).extractall(WORK)
# prompt đã sửa trên Colab (nếu có) ghi đè bản trong bundle, để mọi tài khoản dùng cùng một prompt
if "prompt_rules.txt" in api.list_repo_files(HF_REPO):
    shutil.copy(hf_hub_download(HF_REPO, "prompt_rules.txt"), WORK / "main/prompt_rules.txt")
    print("dùng prompt_rules.txt từ repo HF (bản đã sửa)")
print(sorted(str(p.relative_to(WORK)) for p in WORK.rglob("*") if p.is_file()))

In [ ]:
# 4. SỬA PROMPT (tuỳ chọn). Phần luật nằm ở main/prompt_rules.txt; phần còn lại (schema, context, history, trace) do main/agent.py ghép.
RULES_FILE = WORK / "main/prompt_rules.txt"
NEW_RULES = None          # dán chuỗi luật mới vào đây (hoặc sửa file trong panel Files của Colab) rồi đặt PUSH_PROMPT = True
PUSH_PROMPT = False       # True = lưu prompt lên repo HF để các tài khoản khác dùng đúng bản này
if NEW_RULES:
    RULES_FILE.write_text(NEW_RULES, encoding="utf-8")
if PUSH_PROMPT:
    api.upload_file(path_or_fileobj=str(RULES_FILE), path_in_repo="prompt_rules.txt", repo_id=HF_REPO, commit_message="prompt rules")
    print("đã đẩy prompt_rules.txt lên HF. Nhớ FRESH_START = True nếu đã có checkpoint train bằng prompt cũ")
print(RULES_FILE.read_text(encoding="utf-8"))
print("\n>>> Muốn dùng prompt mới khi thi: chép nội dung này về main/prompt_rules.txt trong repo gốc.")

In [ ]:
# 5. Dựng dữ liệu SFT từ prompt hiện tại (tất định: seed cố định, tách val theo id)
r = subprocess.run([sys.executable, "-I", str(WORK / "finetune/build_sft.py")], cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])
r.check_returncode()
DATA_DIR = WORK / "finetune/out"
h = hashlib.sha256()
for n in ("sft_train.jsonl", "sft_val.jsonl"):
    h.update((DATA_DIR / n).read_bytes())
DATA_HASH = h.hexdigest()[:16]
print("DATA_HASH", DATA_HASH)

In [ ]:
# 6. Kiểm tra checkpoint trên HF để resume
CKPT_ROOT = Path("/content/ckpt"); CKPT = CKPT_ROOT / "last-checkpoint"; OUT = Path("/content/lora")
repo_files = api.list_repo_files(HF_REPO)
if FRESH_START and any(f.startswith("last-checkpoint/") for f in repo_files):
    api.delete_folder("last-checkpoint", repo_id=HF_REPO, commit_message="fresh start")
    repo_files = api.list_repo_files(HF_REPO)
shutil.rmtree(CKPT_ROOT, ignore_errors=True)
RESUME = None
if any(f.startswith("last-checkpoint/") for f in repo_files):
    snapshot_download(HF_REPO, allow_patterns="last-checkpoint/*", local_dir=str(CKPT_ROOT))
    meta = json.loads((CKPT / "sync_meta.json").read_text())
    if meta["data_hash"] != DATA_HASH or meta["train_cfg"] != TRAIN_CFG:
        raise SystemExit("Checkpoint trên HF được train với prompt/dữ liệu/tham số KHÁC bản hiện tại.\n"
                         f"  checkpoint: data={meta['data_hash']} cfg={meta['train_cfg']}\n  hiện tại : data={DATA_HASH} cfg={TRAIN_CFG}\n"
                         "Trả prompt/tham số về như cũ để train tiếp, hoặc đặt FRESH_START = True để train lại từ đầu.")
    RESUME = str(CKPT)
    print(f"RESUME từ step {meta['global_step']} (lưu lúc {meta['saved_at']})")
else:
    print("Chưa có checkpoint: train từ đầu")

In [ ]:
# 7. Token hoá: loss chỉ tính trên phần completion
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type"
print("GPU:", torch.cuda.get_device_name(0))

tok = AutoTokenizer.from_pretrained(MODEL_ID)
def encode(row):
    p = tok.apply_chat_template([{"role": "user", "content": row["prompt"]}], tokenize=False,
                                add_generation_prompt=True, enable_thinking=False)
    p_ids = tok(p, add_special_tokens=False)["input_ids"]
    c_ids = tok(row["completion"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    return {"input_ids": p_ids + c_ids, "labels": [-100] * len(p_ids) + c_ids}

def load(name):
    rows = [json.loads(l) for l in open(DATA_DIR / f"{name}.jsonl", encoding="utf-8")]
    enc = [encode(r) for r in rows]
    kept = [e for e in enc if len(e["input_ids"]) <= MAX_LEN]
    print(name, len(rows), "->", len(kept), "| max len", max(len(e["input_ids"]) for e in enc))
    return kept
train_ds = load("sft_train")
if MAX_EXAMPLES:
    random.seed(0); random.shuffle(train_ds); train_ds = train_ds[:MAX_EXAMPLES]   # seed cố định: mọi tài khoản ra cùng thứ tự

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch); pad = tok.pad_token_id
    ids = torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch])
    lab = torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch])
    return {"input_ids": ids, "labels": lab,
            "attention_mask": (torch.arange(n) < torch.tensor([[len(b["input_ids"])] for b in batch])).long()}

In [ ]:
# 8. Model 4-bit + LoRA (fp16 cố định để resume được giữa T4/L4/A100)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, attn_implementation="sdpa")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

In [ ]:
# 9. Train + tự đồng bộ checkpoint lên HF
T0 = time.time()
TOTAL_STEPS = math.ceil(len(train_ds) / ACCUM) * EPOCHS

class TimeLimit(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if time.time() - T0 > TIME_LIMIT_H * 3600:
            control.should_save = True; control.should_training_stop = True

class HubSync(TrainerCallback):
    # Sau mỗi lần Trainer lưu checkpoint: đẩy cả thư mục lên HF (một commit nguyên tử, thay thế last-checkpoint cũ).
    def on_save(self, args, state, control, **kw):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not ckpt.exists():
            return
        (ckpt / "sync_meta.json").write_text(json.dumps({
            "global_step": state.global_step, "data_hash": DATA_HASH, "train_cfg": TRAIN_CFG,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}))
        for attempt in range(3):
            try:
                api.upload_folder(folder_path=str(ckpt), path_in_repo="last-checkpoint", repo_id=HF_REPO,
                                  delete_patterns=["*"], commit_message=f"step {state.global_step}")
                print(f"[sync] step {state.global_step} -> {HF_REPO}/last-checkpoint", flush=True); return
            except Exception as e:
                print(f"[sync] lỗi lần {attempt + 1}: {e}", flush=True); time.sleep(15)
        print("[sync] KHÔNG đẩy được checkpoint lên HF; checkpoint chỉ còn trên đĩa Colab", flush=True)

args = TrainingArguments(
    output_dir=str(OUT), per_device_train_batch_size=1, gradient_accumulation_steps=ACCUM,
    num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=max(1, int(0.03 * TOTAL_STEPS)),
    fp16=True, gradient_checkpointing=True, logging_steps=5, seed=42, data_seed=42,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2, eval_strategy="no",
    report_to="none", remove_unused_columns=False, dataloader_pin_memory=False)

class CompletionTrainer(Trainer):
    # Chỉ tính logits cho token completion (cuối chuỗi) để tránh bảng logits [seq x vocab] khổng lồ.
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        assert labels.shape[0] == 1, "script giả định batch size 1"
        n_comp = int((labels[0] != -100).sum())
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], logits_to_keep=n_comp + 1)
        logits = out.logits[:, :-1].float(); tgt = labels[:, -n_comp:]
        if num_items_in_batch is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), reduction="sum") / num_items_in_batch
        else:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
        return (loss, out) if return_outputs else loss

trainer = CompletionTrainer(model=model, args=args, train_dataset=train_ds, data_collator=collate,
                            callbacks=[TimeLimit(), HubSync()])
print("tổng step:", TOTAL_STEPS, "| resume:", RESUME)
trainer.train(resume_from_checkpoint=RESUME)

In [ ]:
# 10. Kết thúc: chỉ lưu adapter cuối lên HF khi đã chạy đủ step
if trainer.state.global_step >= trainer.state.max_steps:
    model.save_pretrained(OUT / "final"); tok.save_pretrained(OUT / "final")
    api.upload_folder(folder_path=str(OUT / "final"), path_in_repo="final", repo_id=HF_REPO, delete_patterns=["*"], commit_message="final adapter")
    print("XONG. Adapter cuối ở", f"{HF_REPO}/final")
else:
    print(f"Mới tới step {trainer.state.global_step}/{trainer.state.max_steps}. Chạy lại notebook (tài khoản nào cũng được) để train tiếp.")

In [ ]:
# 11. (Tuỳ chọn) Đo nhanh trên tập val: khớp đúng quyết định kế tiếp (một bước, chưa phải điểm chấm thật)
EVAL_N = 40
model.eval()
val = [json.loads(l) for l in open(DATA_DIR / "sft_val.jsonl", encoding="utf-8")]
random.seed(1); random.shuffle(val)
exact = same_name = bad_json = n = 0
for row in val:
    text = tok.apply_chat_template([{"role": "user", "content": row["prompt"]}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inp = tok(text, return_tensors="pt", add_special_tokens=False)
    if inp["input_ids"].shape[1] > MAX_LEN: continue
    with torch.inference_mode():
        gen = model.generate(**inp.to("cuda"), max_new_tokens=256, do_sample=False)
    pred = tok.decode(gen[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gold = json.loads(row["completion"]); n += 1
    try:
        p = json.loads(pred)
    except ValueError:
        bad_json += 1; continue
    exact += p == gold; same_name += p.get("name") == gold.get("name") and p.get("action") == gold.get("action")
    if n >= EVAL_N: break
print(f"n={n} exact={exact/n:.2f} đúng tool/action={same_name/n:.2f} JSON hỏng={bad_json}")

In [ ]:
# 12. (Tuỳ chọn) Gộp LoRA vào model gốc fp16 để chạy offline trong Docker, rồi đẩy lên repo HF PRIVATE khác (~8 GB).
# Cần RAM CPU ~16 GB; chạy sau cell 10, ở phiên mới nếu cần.
DO_MERGE = False
if DO_MERGE:
    from peft import PeftModel
    adapter = snapshot_download(HF_REPO, allow_patterns="final/*", local_dir="/content/adapter")
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": "cpu"})
    merged = PeftModel.from_pretrained(base, "/content/adapter/final").merge_and_unload()
    merged.save_pretrained("/content/merged"); tok.save_pretrained("/content/merged")
    MERGED_REPO = HF_REPO + "-merged"
    api.create_repo(MERGED_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path="/content/merged", repo_id=MERGED_REPO)
    print("merged ->", MERGED_REPO)